In [2]:
import matplotlib.pyplot as plt
import numpy as np
from qutip import spin_Jx, spin_Jy, spin_Jz, spin_coherent
from scipy.optimize import minimize
from joblib import Parallel, delayed
import time as t
from scipy.linalg import sqrtm, inv, eigh, expm, expm_frechet
import math
import gc

In [ ]:

#Parameters
rng = np.random.default_rng()
J = 4.5
dim = int(2*J+1)
t_steps = 7000
delta_t = 0.001
w_true = rng.normal(0, 0.1, size=3) #True parameter values
w_init = [0, 0, 0] #Initial guess for parameters
batch_size = 50 #Number of parallel optimizations
iter = 500 #Total number of optimization iterations
sn_sd = 1/np.sqrt(100000 * delta_t) #Standard deviation of shot noise
gtol = 1e-5 #Gradient tolerance for optimization convergence
N = 20 #Number of initial states
J_x = spin_Jx(J).full()
J_y = spin_Jy(J).full()
J_z = spin_Jz(J).full()
J_z2 = J_z @ J_z
obs_0 = J_y #Initial observable

if np.array_equal(obs_0, J_x):
    obs_0_string = 'J_x'
elif np.array_equal(obs_0, J_y):
    obs_0_string = 'J_y'
elif np.array_equal(obs_0, J_z):
    obs_0_string = 'J_z'
else:
    obs_0_string = 'unknown'

def random_bloch_directions(N):
    phi = rng.uniform(0, 2*np.pi, N)
    u = rng.uniform(-1, 1, N)
    theta = np.arccos(u)
    return theta, phi

def fibonacci_sphere(N):
    i = np.arange(N)
    phi = 2 * np.pi * (i / ((1 + np.sqrt(5)) / 2) % 1)
    z = 1 - 2*(i + 0.5)/N
    theta = np.arccos(z)
    return theta, phi

theta_rand, phi_rand= random_bloch_directions(N)
theta_fib, phi_fib = fibonacci_sphere(N)
evolution_time = [t_step * delta_t for t_step in range(t_steps+1)]
init_angles  = [(0,0)]
init_rhos = [spin_coherent(J, theta, phi, type = 'dm').full() for theta, phi in zip(theta_rand, phi_rand)]

def commutator(A, B):
   """Compute the commutator [A, B] = AB - BA."""
   return A @ B - B @ A

def expect(obs, state):
    """Compute the expectation value of an observable."""
    return np.real(np.trace(obs @ state))

def fidelity(rho, sigma):
    # Compute sqrt(rho)
    sqrt_rho = sqrtm(rho)
    # Compute sqrt_rho * sigma * sqrt_rho
    inner = sqrt_rho @ sigma @ sqrt_rho
    # Compute sqrt of the inner matrix
    sqrt_inner = sqrtm(inner)
    # Take the trace and square the result
    fidelity_value = np.real(np.trace(sqrt_inner)) ** 2
    return fidelity_value

def compute_expectations(w, init_rhos):
    H_w = w[0]*J_x + w[1]*J_y + w[2]*J_z
    U = expm(-1j * delta_t * H_w)
    U_dag = U.conj().transpose()

    # Frechet derivatives of U
    delU_x = expm_frechet(-1j * delta_t * H_w, -1j * delta_t * J_x, compute_expm=False)
    delU_y = expm_frechet(-1j * delta_t * H_w, -1j * delta_t * J_y, compute_expm=False)
    delU_z = expm_frechet(-1j * delta_t * H_w, -1j * delta_t * J_z, compute_expm=False)

    # Derivatives of U^\dagger
    delUdag_x = delU_x.conj().transpose()
    delUdag_y = delU_y.conj().transpose()
    delUdag_z = delU_z.conj().transpose()

    n_states = len(init_rhos)
    steps_per_state = t_steps // n_states

    expectation = np.zeros(t_steps + 1)
    del_expect_x = np.zeros(t_steps + 1)
    del_expect_y = np.zeros(t_steps + 1)
    del_expect_z = np.zeros(t_steps + 1)

    t_global = 0

    for rho0 in init_rhos:
        rho = rho0.copy()

        del_rho_x = np.zeros_like(rho)
        del_rho_y = np.zeros_like(rho)
        del_rho_z = np.zeros_like(rho)

        # initial expectation
        expectation[t_global] = np.real(np.trace(rho @ obs_0))
        del_expect_x[t_global] = np.real(np.trace(del_rho_x @ obs_0))
        del_expect_y[t_global] = np.real(np.trace(del_rho_y @ obs_0))
        del_expect_z[t_global] = np.real(np.trace(del_rho_z @ obs_0))

        for _ in range(steps_per_state):
            t_global += 1

            rho_new = U @ rho @ U_dag

            del_rho_x_new = (
                delU_x @ rho @ U_dag
                + U @ del_rho_x @ U_dag
                + U @ rho @ delUdag_x
            )

            del_rho_y_new = (
                delU_y @ rho @ U_dag
                + U @ del_rho_y @ U_dag
                + U @ rho @ delUdag_y
            )

            del_rho_z_new = (
                delU_z @ rho @ U_dag
                + U @ del_rho_z @ U_dag
                + U @ rho @ delUdag_z
            )

            rho = rho_new
            del_rho_x = del_rho_x_new
            del_rho_y = del_rho_y_new
            del_rho_z = del_rho_z_new

            expectation[t_global] = np.real(np.trace(rho @ obs_0))
            del_expect_x[t_global] = np.real(np.trace(del_rho_x @ obs_0))
            del_expect_y[t_global] = np.real(np.trace(del_rho_y @ obs_0))
            del_expect_z[t_global] = np.real(np.trace(del_rho_z @ obs_0))

    return expectation, del_expect_x, del_expect_y, del_expect_z

def obj(w, M_vec, init_rhos):
    H_w = w[0]*J_x + w[1]*J_y + w[2]*J_z
    U = expm(-1j * delta_t * H_w)
    U_dag = U.conj().transpose()
    t_global = 0
    expectation = np.zeros(t_steps + 1)
    n_states = len(init_rhos)
    steps_per_state = t_steps // n_states

    for rho0 in init_rhos:
        rho = rho0.copy()

        # initial expectation
        expectation[t_global] = np.real(np.trace(rho @ obs_0))

        for _ in range(steps_per_state):
            t_global += 1
            rho_new = U @ rho @ U_dag
            rho = rho_new
            expectation[t_global] = np.real(np.trace(rho @ obs_0))
    return np.linalg.norm(M_vec - expectation)


def w_estimate(gaussian_noise, expectation, w_init):
    M_vec = expectation + gaussian_noise
    optimization = minimize(obj, x0=w_init, args = (M_vec, init_rhos), method='BFGS', options={'gtol': gtol})
    return optimization


In [14]:
np.arange(4, 40, 4)

array([ 4,  8, 12, 16, 20, 24, 28, 32, 36])

In [ ]:
runs = 1
runtime_array = np.zeros(runs)
covar_array = [None] * runs
mean_array = [None] * runs
std_array = [None] * runs
fisher_array = [None] * runs
w_est_all_runs = [None] * runs
w_est_dict_failures = [None] * runs
bias = [None] * runs
CRB_diff_eig = [None] * runs
fisher_shannon_entropy = [None] * runs

for k in range(runs):
    start_time = t.time()
    shannon = []
    w_est = []
    failures = []

    #Calculate expectation value
    expectation, del_expect_x, del_expect_y, del_expect_z = compute_expectations(w_true)
    
    #Calculate fisher information matrix and shannon entropy
    for j in np.arange(0, t_steps+1, 10):
        del_expectation_x = del_expect_x[:j]
        del_expectation_y = del_expect_y[:j]
        del_expectation_z = del_expect_z[:j]
        fisher_matrix = np.zeros((3, 3))
        fisher_matrix[0, 0] = np.vdot(del_expectation_x, del_expectation_x) / sn_sd**2
        fisher_matrix[1, 1] = np.vdot(del_expectation_y, del_expectation_y) / sn_sd**2
        fisher_matrix[2, 2] = np.vdot(del_expectation_z, del_expectation_z) / sn_sd**2
        fisher_matrix[0, 1] = np.vdot(del_expectation_x, del_expectation_y) / sn_sd**2
        fisher_matrix[0, 2] = np.vdot(del_expectation_x, del_expectation_z) / sn_sd**2
        fisher_matrix[1, 2] = np.vdot(del_expectation_y, del_expectation_z) / sn_sd**2
        fisher_matrix[1, 0] = fisher_matrix[0, 1]
        fisher_matrix[2, 0] = fisher_matrix[0, 2]
        fisher_matrix[2, 1] = fisher_matrix[1, 2]
        fisher_eig = np.linalg.eigvals(fisher_matrix)
        fisher_eig = fisher_eig/np.sum(fisher_eig)
        shannon.append(-np.sum(fisher_eig * np.log(fisher_eig)))
    fisher_shannon_entropy[k] = np.array(shannon)
    fisher_array[k] = fisher_matrix

    while(len(w_est) < iter):
        noise = [rng.normal(0, sn_sd, t_steps+1) for _ in range(batch_size)]
        w_est_dict = Parallel(n_jobs=-1)(delayed(w_estimate)(gaussian_noise = n, expectation = expectation, w_init = w_init) for n in noise)
        #failures.extend([res for res in w_est_dict if not res.success])
        successes = [res.x for res in w_est_dict if res.success]
        w_est.extend(successes)
    #w_est_dict_failures[k] = failures
    w_est = np.array(w_est[:iter])
    #w_est_all_runs[k] = w_est

   #Calculate covariance and mean of w estimates
    covariance = np.cov(w_est, rowvar=False)
    covar_array[k] = covariance
    mean_array[k] = np.mean(w_est, axis=0)
    bias[k] = mean_array[k] - np.array(w_true)
    std_array[k] = np.diag(covariance)

    end_time = t.time()
    runtime_array[k] = end_time - start_time
    
fisher_shannon_time = delta_t * np.arange(0, t_steps+1, 10)
fisher_inv_array = np.linalg.inv(fisher_array)
fisher_inv_trace = np.trace(fisher_inv_array, axis1=1, axis2=2)
fisher_inv_normalized = fisher_inv_array / fisher_inv_trace[:, None, None]
cov_array_trace = np.trace(covar_array, axis1=1, axis2=2)
cov_array_normalized = covar_array / cov_array_trace[:, None, None]
infidelity_values = np.array([1-fidelity(fisher_inv_normalized[i], cov_array_normalized[i]) for i in range(runs)])
MSE = np.array([[bias[i][k]**2 + std_array[i][k] for k in range(len(w_true))] for i in range(runs)])

for i in range(runs):
    CRB_diff = covar_array[i] - fisher_inv_array[i]
    CRB_diff_eig[i] = np.linalg.eigvals(CRB_diff)


C:\Users\alexg\AppData\Local\Temp\ipykernel_22416\1805041961.py:92: RuntimeWarning: invalid value encountered in divide
  fisher_eig = fisher_eig/np.sum(fisher_eig)


In [ ]:
#Folder to save data in
folder = "/users/agarcia2001/data_"

parameters = (
    f"Parameters: J = {J}; w = {w_true}; sn_sd = {sn_sd:.3f}; "
    f"delta_t = {delta_t}; t_steps = {t_steps}; iter = {iter}; "
    f"Initial state: theta_scs = {theta_scs:.3f}; phi_scs = {phi_scs:.3f}; Initial observable = {obs_0_string}"
)


np.save(f"{folder}/mean_array.npy", mean_array)
np.save(f"{folder}/bias.npy", bias)
np.save(f"{folder}/std_array.npy", std_array)
np.save(f"{folder}/covar_array.npy", covar_array)
np.save(f"{folder}/fisher.npy", fisher_array)
np.save(f"{folder}/runtime.npy", runtime_array)
#np.save(f"{folder}/w_est_all_runs.npy", np.array(w_est_all_runs))
np.save(f"{folder}/fidelity.npy", infidelity_values)
np.save(f"{folder}/evolution_time.npy", evolution_time)
np.save(f"{folder}/fisher_inv_trace.npy", fisher_inv_trace)
np.save(f"{folder}/covar_trace.npy", cov_array_trace)
np.save(f"{folder}/CRB_diff_eig.npy", CRB_diff_eig)
np.save(f"{folder}/fisher_shannon_entropy.npy", fisher_shannon_entropy)
np.save(f"{folder}/fisher_shannon_time.npy", fisher_shannon_time)
with open(f"{folder}/parameters.txt", "w") as f:
    f.write(parameters)
# with open(f"{folder}/w_est_failures.txt", "w") as f:
#     f.write(str(w_est_dict_failures))



